# Lab 7: ABC del aprendizaje de máquina

Dataset: California Housing (censo 1990). El objetivo es entender el dataset desde cero,
evaluar su calidad, hacer EDA y extraer primeras conclusiones antes de modelar.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (8, 4)

## 1. Carga y primera inspección

Cargamos el CSV directamente desde el repositorio del curso y revisamos las primeras filas para tener una idea de la estructura.

In [ ]:
URL = 'https://raw.githubusercontent.com/hernansalinas/Curso_aprendizaje_estadistico/main/datasets/Sesion_07_housing.csv'
df = pd.read_csv(URL)

print('Filas x columnas:', df.shape)
df.head()

## 2. Calidad de datos

Revisamos tipos, valores faltantes y rango de cada columna.

In [ ]:
# Tipos de dato y conteo de no-nulos
df.info()

In [ ]:
# Estadísticas descriptivas básicas
df.describe()

In [ ]:
# Valores faltantes por columna
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

resumen_faltantes = pd.DataFrame({'faltantes': missing, 'porcentaje_%': missing_pct})
display(resumen_faltantes[resumen_faltantes['faltantes'] > 0])

print(f'\nTotal de filas: {len(df)}')
print('Solo total_bedrooms tiene valores faltantes. Son pocos, se pueden imputar o descartar.')

### Punto 3: Variables y tipos

Hay 9 columnas numéricas y 1 categórica (`ocean_proximity`). Las numéricas representan cantidades geográficas,
demográficas y económicas del distrito. La variable objetivo es `median_house_value`.

In [ ]:
print('Tipos de datos:')
print(df.dtypes)
print(f'\nNúmero de columnas numéricas: {df.select_dtypes(include=np.number).shape[1]}')
print(f'Número de columnas categóricas: {df.select_dtypes(include="object").shape[1]}')

### Punto 5: Elementos únicos en ocean_proximity

In [ ]:
print('Categorías únicas de ocean_proximity:')
print(df['ocean_proximity'].unique())
print()
print('Conteo por categoría:')
display(df['ocean_proximity'].value_counts().to_frame())

### Punto 6: Promedios por ocean_proximity (groupby)

Para cada categoría de proximidad al océano calculamos el promedio de las columnas numéricas.
Esto da una primera idea de qué tan diferentes son los distritos según su ubicación costera.

In [ ]:
cols = ['housing_median_age', 'total_rooms', 'total_bedrooms',
        'population', 'households', 'median_income', 'median_house_value']

promedios = df.groupby('ocean_proximity')[cols].mean().round(1)
display(promedios)

print('\nNota: los distritos ISLAND tienen el mayor ingreso medio y valor de vivienda.')

### Punto 7: Histogramas de cada columna numérica

Revisamos la distribución de cada variable para detectar asimetrías, colas largas o valores acumulados en el límite.

In [ ]:
cols_num = df.select_dtypes(include=np.number).columns.tolist()

fig, axes = plt.subplots(3, 3, figsize=(13, 9))
axes = axes.ravel()

for i, col in enumerate(cols_num):
    axes[i].hist(df[col].dropna(), bins=40, color='steelblue', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')
    axes[i].grid(alpha=0.2)

plt.suptitle('Distribución de variables numéricas', y=1.01)
plt.tight_layout()
plt.show()

print('Observaciones:')
print('- median_house_value tiene un pico artificial en 500 000: el dataset tiene el valor máximo capado.')
print('- median_income y housing_median_age tienen distribuciones relativamente razonables.')
print('- total_rooms, total_bedrooms, population y households tienen colas derechas largas (outliers).')

## 3. Diagramas de caja e identificación de outliers

El boxplot resume la distribución en cuartiles y marca como outliers los puntos fuera de
$Q_1 - 1.5 \cdot IQR$ y $Q_3 + 1.5 \cdot IQR$.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 9))
axes = axes.ravel()

for i, col in enumerate(cols_num):
    axes[i].boxplot(df[col].dropna(), vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='red', linewidth=1.5))
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xticks([])
    axes[i].grid(alpha=0.2)

plt.suptitle('Diagramas de caja - variables numéricas', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Conteo de outliers por columna según criterio IQR
def contar_outliers(serie):
    s = serie.dropna()
    Q1 = s.quantile(0.25)
    Q3 = s.quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    return ((s < lim_inf) | (s > lim_sup)).sum()

outliers_df = pd.DataFrame({
    'outliers_IQR': {col: contar_outliers(df[col]) for col in cols_num},
    'porcentaje_%': {col: round(contar_outliers(df[col]) / len(df) * 100, 2) for col in cols_num}
})

display(outliers_df.sort_values('outliers_IQR', ascending=False))
print('\ntotal_rooms, population y households tienen el mayor porcentaje de outliers: distritos muy grandes o muy pequeños.')

### Punto 8: Boxplot del precio por ocean_proximity

Visualizamos la distribución del valor de vivienda para cada categoría de proximidad al océano.

In [ ]:
categorias = df['ocean_proximity'].value_counts().index.tolist()
datos_box = [df.loc[df['ocean_proximity'] == cat, 'median_house_value'].values for cat in categorias]

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(datos_box, labels=categorias, patch_artist=True,
                medianprops=dict(color='red', linewidth=1.5))
colores = ['steelblue', 'darkorange', 'green', 'mediumorchid', 'crimson']
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_ylabel('Valor mediano de vivienda (USD)')
ax.set_title('Distribución del precio de vivienda por proximidad al océano')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print('INLAND tiene los precios más bajos. ISLAND y NEAR BAY los más altos.')
print('Todos presentan outliers hacia arriba: algunos distritos muy caros en cada categoría.')

## 4. Análisis exploratorio adicional (Puntos 10-11)

Revisamos correlaciones entre variables numéricas y la variable objetivo,
y visualizamos la relación entre features con color por categoría de proximidad al océano.

In [ ]:
# Matriz de correlación
corr = df[cols_num].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, annot_kws={'size': 8})
plt.title('Matriz de correlación (Pearson)')
plt.tight_layout()
plt.show()

print('median_income tiene la correlación más alta con median_house_value (0.69 aprox).')
print('total_rooms, households y population están altamente correlacionadas entre sí (multicolinealidad).')

In [ ]:
# Pairplot de las features más relevantes (Punto 10)
cols_pairplot = ['median_income', 'housing_median_age', 'total_rooms', 'median_house_value']
muestra = df[cols_pairplot].sample(500, random_state=42)

g = sns.pairplot(muestra, diag_kind='hist',
                 plot_kws={'alpha': 0.3, 's': 10},
                 diag_kws={'bins': 25})
g.figure.suptitle('Pairplot de features principales (muestra de 500 distritos)', y=1.02)
plt.show()

print('La relación más lineal es entre median_income y median_house_value.')

In [ ]:
# Mapa geográfico de precios (Punto 11)
plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['median_house_value'],
    cmap='plasma', alpha=0.4, s=df['population'] / 100,
    vmin=df['median_house_value'].quantile(0.02),
    vmax=df['median_house_value'].quantile(0.98)
)
plt.colorbar(scatter, label='Valor mediano de vivienda (USD)')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Distribución geográfica del precio de vivienda en California\n(tamaño del punto proporcional a la población)')
plt.grid(alpha=0.2)
plt.show()

print('Se ven claramente los precios altos en la costa: bahía de San Francisco y Los Ángeles.')

In [ ]:
# Relación entre ingreso medio y precio de vivienda, coloreado por categoría
cats_ocean = df['ocean_proximity'].unique()
paleta = sns.color_palette('Set2', n_colors=len(cats_ocean))
color_map = dict(zip(cats_ocean, paleta))

plt.figure(figsize=(8, 5))
for cat in cats_ocean:
    mask = df['ocean_proximity'] == cat
    plt.scatter(df.loc[mask, 'median_income'], df.loc[mask, 'median_house_value'],
                alpha=0.15, s=6, color=color_map[cat], label=cat)
plt.xlabel('Ingreso mediano (x$10^3$)')
plt.ylabel('Valor mediano de vivienda (USD)')
plt.title('Ingreso vs Valor de vivienda por categoría de ocean_proximity')
plt.legend(markerscale=4, fontsize=9)
plt.grid(alpha=0.2)
plt.show()

print('Se ve una tendencia positiva clara, aunque con mucha dispersión.')
print('El techo de 500 000 USD es claramente visible como una línea horizontal de puntos.')

## 5. División estratificada del dataset (Punto 12)

Para que el conjunto de entrenamiento y de prueba representen bien la distribución de ingresos,
usamos `StratifiedShuffleSplit` con una variable auxiliar `income_cat` basada en `median_income`.

La idea es que si hay pocos distritos de ingresos muy altos, el split aleatorio podría excluirlos del test.

In [ ]:
# Crear income_cat para estratificar
df['income_cat'] = pd.cut(df['median_income'],
                           bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
                           labels=[1, 2, 3, 4, 5])

plt.figure(figsize=(7, 4))
df['income_cat'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.xlabel('Categoría de ingreso')
plt.ylabel('Cantidad de distritos')
plt.title('Distribución de income_cat')
plt.xticks(rotation=0)
plt.grid(alpha=0.2)
plt.show()

# Split estratificado
split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_idx, test_idx in split.split(df, df['income_cat']):
    strat_train = df.iloc[train_idx].copy()
    strat_test  = df.iloc[test_idx].copy()

# Comparar distribuciones
comparacion = pd.DataFrame({
    'Global': df['income_cat'].value_counts(normalize=True).sort_index(),
    'Train':  strat_train['income_cat'].value_counts(normalize=True).sort_index(),
    'Test':   strat_test['income_cat'].value_counts(normalize=True).sort_index()
})
display(comparacion.round(3))
print('La distribución de income_cat es prácticamente idéntica en train y test: el split estratificado funciona.')

# Limpiar la columna auxiliar
for df_s in [df, strat_train, strat_test]:
    df_s.drop('income_cat', axis=1, inplace=True)

print(f'\nTrain: {strat_train.shape}, Test: {strat_test.shape}')

## 6. Ingeniería de características (Punto 13)

Combinamos features existentes para crear nuevas variables más informativas.
Las variables de conteo total (total_rooms, total_bedrooms) no son tan útiles como sus ratios por hogar.

In [ ]:
def agregar_features(df_in):
    df_out = df_in.copy()
    df_out['rooms_per_household']      = df_out['total_rooms'] / df_out['households']
    df_out['bedrooms_per_room']        = df_out['total_bedrooms'] / df_out['total_rooms']
    df_out['population_per_household'] = df_out['population'] / df_out['households']
    return df_out

strat_train = agregar_features(strat_train)
strat_test  = agregar_features(strat_test)

# Correlación de las nuevas features con el precio
nuevas_cols = ['median_house_value', 'median_income',
               'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
corr_nueva = strat_train[nuevas_cols].corr()[['median_house_value']].round(3)
display(corr_nueva)

print('rooms_per_household tiene correlación positiva con el precio.')
print('bedrooms_per_room tiene correlación negativa: más dormitorios por cuarto -> más barato.')

## 7. Pipeline de preparación de datos (Puntos 14-17)

Construimos un pipeline con `sklearn` que aplica en orden:
- `SimpleImputer` (mediana) para los valores faltantes en `total_bedrooms`
- `MinMaxScaler` para normalizar las features numéricas
- `OneHotEncoder` para convertir `ocean_proximity` en variables dummy

El `ColumnTransformer` combina los dos sub-pipelines (numérico y categórico) automáticamente.

In [ ]:
# Separar features y etiqueta
X_train = strat_train.drop('median_house_value', axis=1)
y_train = strat_train['median_house_value'].copy()

X_test  = strat_test.drop('median_house_value', axis=1)
y_test  = strat_test['median_house_value'].copy()

# Identificar columnas numéricas y categóricas
cols_num_prep = X_train.select_dtypes(include=[np.number]).columns.tolist()
cols_cat_prep = ['ocean_proximity']

print(f'Features numéricas ({len(cols_num_prep)}): {cols_num_prep}')
print(f'Features categóricas: {cols_cat_prep}')

In [ ]:
# Pipeline para features numéricas: imputer + scaler
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),   # rellena total_bedrooms faltantes
    ('scaler',  MinMaxScaler()),                      # normaliza a [0, 1]
])

# Pipeline para features categóricas: one-hot encoding
pipeline_cat = Pipeline([
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore')),
])

# ColumnTransformer combina ambos pipelines
preprocesador = ColumnTransformer([
    ('num', pipeline_num, cols_num_prep),
    ('cat', pipeline_cat, cols_cat_prep),
])

# Fit en train, transform en ambos
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep  = preprocesador.transform(X_test)

# Nombres de features resultantes
cat_names = list(preprocesador.named_transformers_['cat']
                 .named_steps['encoder']
                 .get_feature_names_out(['ocean_proximity']))
feature_names = cols_num_prep + cat_names

print(f'X_train_prep shape: {X_train_prep.shape}')
print(f'X_test_prep  shape: {X_test_prep.shape}')
print(f'\nFeatures finales ({len(feature_names)}):')
print(feature_names)

# Mostrar primeras filas transformadas
df_prep = pd.DataFrame(X_train_prep[:5], columns=feature_names)
display(df_prep.round(3))

In [ ]:
# Verificar que la imputación funcionó (no deben quedar NaN)
print(f'NaN en X_train_prep: {np.isnan(X_train_prep).sum()}')
print(f'NaN en X_test_prep:  {np.isnan(X_test_prep).sum()}')
print()
print(f'Rango de X_train_prep (parte numérica): [{X_train_prep[:, :len(cols_num_prep)].min():.3f}, {X_train_prep[:, :len(cols_num_prep)].max():.3f}]')
print('Los valores numéricos están en [0, 1]: MinMaxScaler funcionó correctamente.')
print()
print('Las columnas categóricas (ocean_proximity) se convierten en 5 columnas dummy 0/1.')

## Conclusiones

- El dataset tiene 20 640 observaciones y 10 columnas. Solo `total_bedrooms` tiene valores faltantes (~1%).
- `ocean_proximity` es la única variable categórica; los distritos INLAND son los más numerosos.
- `median_income` es el predictor más correlacionado con el precio de vivienda.
- Hay un límite artificial en `median_house_value` a los 500 000 USD que habrá que tener en cuenta en el modelado.
- El split estratificado por `income_cat` asegura que train y test representen bien la distribución de ingresos.
- Las features derivadas (`rooms_per_household`, `bedrooms_per_room`) son más informativas que los conteos absolutos.
- El pipeline con `SimpleImputer` + `MinMaxScaler` + `OneHotEncoder` prepara el dataset listo para cualquier modelo sklearn.